SHAP explainer

In [1]:
import os
os.chdir("..")  # run from the repo root so artifacts/ and data/ paths resolve

import pandas as pd
import shap
from src.data.ingestion import DataIngestion
from src.data.preprocessing import DataPreprocessing
from src.utils.helper import load_object

c:\Users\aishv\Documents\github_projects\churn-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_path, val_path, test_path = DataIngestion().initiate_data_ingestion()
X_train, X_val, X_test, y_train, y_val, y_test = (
    DataPreprocessing().initiate_data_preprocessing(train_path, val_path, test_path)
)

model = load_object("artifacts/model.pkl")
preprocessor = load_object("artifacts/preprocessor.pkl")
feature_names = preprocessor.get_feature_names_out()

In [3]:
X_sample = pd.DataFrame(X_test[:100], columns=feature_names)

In [4]:
feature_names

array(['numeric__tenure', 'numeric__MonthlyCharges',
       'numeric__TotalCharges', 'binary__SeniorCitizen',
       'binary__Partner', 'binary__Dependents', 'binary__PhoneService',
       'binary__PaperlessBilling', 'binary__MultipleLines',
       'binary__OnlineSecurity', 'binary__OnlineBackup',
       'binary__DeviceProtection', 'binary__TechSupport',
       'binary__StreamingTV', 'binary__StreamingMovies',
       'nominal__gender_Male', 'nominal__InternetService_Fiber optic',
       'nominal__InternetService_No', 'nominal__Contract_One year',
       'nominal__Contract_Two year',
       'nominal__PaymentMethod_Credit card (automatic)',
       'nominal__PaymentMethod_Electronic check',
       'nominal__PaymentMethod_Mailed check'], dtype=object)

SHAP explainer

In [ ]:
explainer = shap.Explainer(model)
shap_values = explainer(X_sample)
print(shap_values.shape)    # (100, n_features, 2)

(100, 23, 2)


In [8]:
sv_churn = shap_values[:, :, 1]     # churn class only

In [10]:
# Check additivity for customer 0: these two numbers should match
print(sv_churn.base_values[0] + sv_churn.values[0].sum())
print(model.predict_proba(X_sample.iloc[[0]])[0, 1])

0.02417604031001902
0.024176040310039837


c:\Users\aishv\Documents\github_projects\churn-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [13]:
shap.plots.waterfall(sv_churn[0])   # why customer 0 got their score
shap.plots.beeswarm(sv_churn)       # what drives churn across 100 customers

ImportError: matplotlib is not installed so plotting is not available! Run `pip install matplotlib` to fix this.